In [16]:
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    raise EnvironmentError("GROQ_API_KEY environment variable is not set.")

In [17]:
llm  = ChatGroq(model="llama-3.1-8b-instant", temperature=0.5)

Lets use the DuckDuckGo seach tool which fetch the realtime data from the internet.

In [1]:
from langchain_community.tools import DuckDuckGoSearchRun

duck_search = DuckDuckGoSearchRun()

duck_search.invoke("latest news on AI")

'Read full articles, watch videos, browse thousands of titles and more on the "Artificial intelligence" topic with Google News. Apr 28, 2026 · Explore the latest artificial intelligence news with Reuters - from AI breakthroughs and technology trends to regulation, ethics, business and global impact. Read the latest on artificial intelligence and machine learning tech, the companies that are building them, and the ethical issues AI raises today. 2 days ago · See the latest on The AI Race. From breaking news to in-depth reporting, Bloomberg tracks the full story in real time. Apr 17, 2026 · Artificial Intelligence News. Everything on AI including futuristic robots with artificial intelligence, computer models of human intelligence and more.'

We need to call with a decorator for better approach

In [3]:
from langchain_core.tools import tool

@tool
def Duck_Search(query: str) -> str:
    '''This function performs a search using the DuckDuckGo search engine and returns the results.'''
    duck_search = DuckDuckGoSearchRun()
    return duck_search.invoke(query)


We can make our CustomTool So

In [6]:
@tool
def custom_tool(query: str) -> str:
    '''This function provide the personal information of the user.'''
    info = {
        "Alice": "Alice is a software engineer with 5 years of experience in machine learning.",
        "Bob": "Bob is a data scientist who specializes in natural language processing.",
        "Charlie": "Charlie is a product manager with a background in AI and cloud computing."
    }
    return info.get(query, "No information available for this user.")

Also use the WikiPedia Tool 

In [12]:
wiki_query = WikipediaQueryRun(api_wrapper= WikipediaAPIWrapper())
wiki_query.invoke("what is langchain?")

'Page: Vector database\nSummary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector databases include similarity search, semantic search, multi-modal search, recommendations engines, object detection, and retrieval-augmented generation (RAG).\nVector embeddings are mathematical representations of data in a high-dimensional space. In this space, each dimension corresponds to a feature of the data, with the number of dimensions ranging from a few hundred to tens of thousands, depending on the complexity of the data being represented. Each data item is represented by one vector in this space. Words, phrases, or entire documents, as well as images, audio, and other typ

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

def wikipedia_search(query: str) -> str:
    '''This function performs a search using the Wikipedia API and returns the results.'''
    wiki_query = WikipediaQueryRun(api_wrapper= WikipediaAPIWrapper())
    return wiki_query.invoke(query)

In [18]:
#Now bind the tools to the LLM in langchain
llm_bind_tool = llm.bind_tools([Duck_Search,custom_tool,wikipedia_search])


In [21]:
#Now invoke
response = llm_bind_tool.invoke("What is the latest news on AI and who is Alice?")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'tj95t1mrz', 'function': {'arguments': '{"query":"latest AI news"}', 'name': 'wikipedia_search'}, 'type': 'function'}, {'id': '1jygph5zj', 'function': {'arguments': '{"query":"Alice AI"}', 'name': 'wikipedia_search'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 372, 'total_tokens': 404, 'completion_time': 0.075397973, 'completion_tokens_details': None, 'prompt_time': 0.050027353, 'prompt_tokens_details': None, 'queue_time': 0.052117156, 'total_time': 0.125425326}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e02b6-5fb3-7660-b05f-a1065b141a0c-0' tool_calls=[{'name': 'wikipedia_search', 'args': {'query': 'latest AI news'}, 'id': 'tj95t1mrz', 'type': 'tool_call'}, {'name': 'wikipedia_search', 'args': {'query': 'Alice AI'}, 'id': '1jygp

We connect the tool and llm using langchain and we gonna use langgraph 